# Hyperparameter Tuning using Grid Search for Support Vector Machines

This notebook supports the tutorial report on **hyperparameter tuning using grid search**.

The aim is to show how **GridSearchCV** can be used to improve the performance of a Support Vector Machine (SVM) by systematically testing different hyperparameter combinations.

### Objectives
- build a baseline SVM model
- apply grid search with cross-validation
- compare baseline and tuned model performance
- visualise decision boundaries
- interpret the effect of hyperparameters such as `C`, `kernel`, and `gamma`

This notebook is structured to be clear, reproducible, and suitable for coursework submission.


## 1. Imports

The following libraries are used for dataset generation, preprocessing, model training, hyperparameter tuning, evaluation, and visualisation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Dataset

A `make_moons` dataset is used because it is non-linear and easy to visualise. This makes it suitable for demonstrating how SVM hyperparameters influence decision boundaries.


In [ ]:
X, y = make_moons(n_samples=600, noise=0.25, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, alpha=0.8)
plt.title("Training Data (make_moons)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()


## 3. Feature Scaling

Since SVM is sensitive to feature magnitudes, standardisation is applied before training.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Baseline SVM Model

Before applying grid search, a baseline SVM model is trained using default parameters. This provides a reference point for comparison.


In [ ]:
baseline_model = SVC(random_state=42)
baseline_model.fit(X_train_scaled, y_train)

baseline_pred = baseline_model.predict(X_test_scaled)
baseline_acc = accuracy_score(y_test, baseline_pred)

print("Baseline Test Accuracy:", round(baseline_acc, 4))
print("\nClassification Report (Baseline):")
print(classification_report(y_test, baseline_pred))


## 5. Hyperparameter Grid

The following parameter grid is used for tuning:

- `C`: controls regularisation
- `kernel`: determines the type of decision boundary
- `gamma`: controls the influence of individual points for the RBF kernel


In [ ]:
param_grid = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", 0.01, 0.1, 1]
}

param_grid


## 6. Grid Search with Cross-Validation

`GridSearchCV` is used to evaluate all combinations of the specified hyperparameters using cross-validation.

This ensures that the selected hyperparameters are based on validation performance rather than training accuracy alone.


In [ ]:
grid_search = GridSearchCV(
    estimator=SVC(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Score:", round(grid_search.best_score_, 4))


## 7. Best Model Evaluation

The best model found by grid search is evaluated on the unseen test set.


In [ ]:
best_model = grid_search.best_estimator_
best_pred = best_model.predict(X_test_scaled)
best_acc = accuracy_score(y_test, best_pred)

print("Tuned Test Accuracy:", round(best_acc, 4))
print("\nClassification Report (Tuned Model):")
print(classification_report(y_test, best_pred))


## 8. Compare Baseline and Tuned Performance

This comparison shows whether hyperparameter tuning improves the final model.


In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Baseline SVM", "Tuned SVM"],
    "Accuracy": [baseline_acc, best_acc]
})

comparison_df


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(comparison_df["Model"], comparison_df["Accuracy"])
plt.ylim(0, 1)
plt.ylabel("Test Accuracy")
plt.title("Baseline vs Tuned SVM Accuracy")
plt.show()


## 9. Grid Search Results Table

The table below summarises the hyperparameter combinations tested during grid search.


In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df[[
    "param_C",
    "param_kernel",
    "param_gamma",
    "mean_test_score",
    "rank_test_score"
]].sort_values("rank_test_score")

results_df.head(10)


## 10. Heatmap of Grid Search Scores

To make the tuning process easier to interpret, a heatmap is created for the RBF kernel.


In [ ]:
rbf_results = results_df[results_df["param_kernel"] == "rbf"].copy()

heatmap_data = rbf_results.pivot_table(
    values="mean_test_score",
    index="param_C",
    columns="param_gamma"
)

plt.figure(figsize=(7, 5))
plt.imshow(heatmap_data, aspect="auto")
plt.colorbar(label="Mean CV Accuracy")
plt.xticks(range(len(heatmap_data.columns)), heatmap_data.columns)
plt.yticks(range(len(heatmap_data.index)), heatmap_data.index)
plt.xlabel("Gamma")
plt.ylabel("C")
plt.title("Grid Search Scores for RBF Kernel")
plt.show()

heatmap_data


## 11. Decision Boundary Function

The following helper function is used to visualise how the baseline and tuned models classify the dataset.


In [ ]:
def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.35)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", s=25)
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")


## 12. Decision Boundary: Baseline Model

In [ ]:
plt.figure(figsize=(6, 5))
plot_decision_boundary(baseline_model, X_test_scaled, y_test, "Baseline SVM Decision Boundary")
plt.show()


## 13. Decision Boundary: Tuned Model

In [ ]:
plt.figure(figsize=(6, 5))
plot_decision_boundary(best_model, X_test_scaled, y_test, "Tuned SVM Decision Boundary")
plt.show()


## 14. Confusion Matrices

Confusion matrices provide additional insight into the quality of the baseline and tuned models.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

cm_baseline = confusion_matrix(y_test, baseline_pred)
cm_tuned = confusion_matrix(y_test, best_pred)

axes[0].imshow(cm_baseline)
axes[0].set_title("Baseline SVM")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

axes[1].imshow(cm_tuned)
axes[1].set_title("Tuned SVM")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()

print("Baseline Confusion Matrix:\n", cm_baseline)
print("\nTuned Confusion Matrix:\n", cm_tuned)


## 15. Interpretation Notes

When writing the report, focus on the following:

- Did grid search improve test accuracy?
- Which hyperparameters were selected as best?
- Did the tuned model produce a better decision boundary?
- What does the heatmap suggest about the interaction between `C` and `gamma`?
- Does the tuned model appear more balanced or more flexible than the baseline?

These observations can be used directly in the experimental and analysis sections of the report.


## 16. References to Mention in the Report

- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.
- Cortes, C., & Vapnik, V. (1995). *Support-Vector Networks*.
- Shalev-Shwartz, S., & Ben-David, S. (2014). *Understanding Machine Learning: From Theory to Algorithms*.
- Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python*.
- Scikit-learn documentation for `GridSearchCV` and `SVC`.
